In [4]:
import json
from pathlib import Path

base_dir = Path('../../data/synflow')
input_path = base_dir / 'fragment_128k_to_real_cost.json'
output_path = base_dir / 'fragment_128k_to_real_cost_sanitized.json'
fragment_to_cost = json.load(open(input_path, 'r'))

In [5]:
import networkx as nx
from rdkit.Chem import SaltRemover, BondType
from rdkit import Chem

class Graph(nx.Graph):
    def __str__(self):
        return repr(self)

    def __repr__(self):
        return f'<{list(self.nodes)}, {list(self.edges)}, {list(self.nodes[i]["v"] for i in self.nodes)}>'

    def bridges(self):
        return list(nx.bridges(self))

    def relabel_nodes(self, rmap):
        return nx.relabel_nodes(self, rmap)

    def clear_cache(self):
        self._Data_cache = None

allow_explicitly_aromatic = True
def obj_to_graph(mol: Chem.Mol) -> Graph:
    """Convert an RDMol to a Graph"""
    g = Graph()
    mol = Chem.Mol(mol)  # Make a copy
    try:
        mol.UpdatePropertyCache()
        if not allow_explicitly_aromatic:
            # If we disallow aromatic bonds, ask rdkit to Kekulize mol and remove aromatic bond flags
            Chem.Kekulize(mol, clearAromaticFlags=True)
    except Exception as e:
        # manually replace "c" with "C" in the smiles string
        mol = Chem.MolFromSmiles(Chem.MolToSmiles(mol).replace("c", "C"))
        mol.UpdatePropertyCache()
    # Only set an attribute tag if it is not the default attribute
    for a in mol.GetAtoms():
        attrs = {
            "atomic_number": a.GetAtomicNum(),
            "chi": a.GetChiralTag(),
            "charge": a.GetFormalCharge(),
            "expl_H": a.GetNumExplicitHs(),
        }
        g.add_node(
            a.GetIdx(),
            v=a.GetSymbol(),
            **{attr: val for attr, val in attrs.items()},
            **({"fill_wildcard": None} if a.GetSymbol() == "*" else {}),
        )
    for b in mol.GetBonds():
        attrs = {"type": b.GetBondType()}
        g.add_edge(
            b.GetBeginAtomIdx(),
            b.GetEndAtomIdx(),
            **{attr: val for attr, val in attrs.items()},
        )
    return g

default_wildcard_replacement = "C"
def graph_to_obj(g: Graph) -> Chem.Mol:
    """Convert a Graph to an RDKit Mol"""
    mp = Chem.RWMol()
    mp.BeginBatchEdit()
    for i in range(len(g.nodes)):
        d = g.nodes[i]
        s = d.get("fill_wildcard", d["v"])
        a = Chem.Atom(s if s is not None else default_wildcard_replacement)
        if "chi" in d:
            a.SetChiralTag(d["chi"])
        if "charge" in d:
            a.SetFormalCharge(d["charge"])
        if "expl_H" in d:
            a.SetNumExplicitHs(d["expl_H"])
        if "no_impl" in d:
            a.SetNoImplicit(d["no_impl"])
        mp.AddAtom(a)
    for e in g.edges:
        d = g.edges[e]
        mp.AddBond(e[0], e[1], d.get("type", BondType.SINGLE))
    mp.CommitBatchEdit()
    try:
        Chem.SanitizeMol(mp)
    except Chem.AtomKekulizeException as e:
        rw_mol = Chem.RWMol(mp)
        for a in rw_mol.GetAtoms():
            if (not a.IsInRing()) and a.GetIsAromatic():
                a.SetIsAromatic(False)
        for b in rw_mol.GetBonds():
            if (not b.IsInRing()) and b.GetIsAromatic():
                b.SetIsAromatic(False)
        mp = Chem.Mol(rw_mol)
    Chem.SanitizeMol(mp)
    return Chem.MolFromSmiles(Chem.MolToSmiles(mp))

def sanitize(smiles):
    mol = Chem.MolFromSmiles(smiles)
    remover = SaltRemover.SaltRemover()
    mol = remover.StripMol(mol)
    Chem.RemoveStereochemistry(mol)
    mol = graph_to_obj(obj_to_graph(mol))

    return Chem.MolToSmiles(mol)


In [6]:
fragment_to_cost_sanitized = {sanitize(k): v for k, v in fragment_to_cost.items()}
json.dump(fragment_to_cost_sanitized, open(output_path, 'w'))